# Chronos 2 Foundation Model Forecasting

This notebook implements zero-shot forecasting for hourly bike rental demand using Chronos 2.

The following Chronos 2 strategies are evaluated:
- Univariate Chronos 2,
- Covariate Chronos 2.

The objective of this notebook is to validate both strategies using the same expanding-window approach as the LightGBM model, select the best setup based on validation MASE, and finally evaluate the selected model on the test set.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [2]:
DATA_DIR = "../data/processed"
FORECAST_DIR = "../outputs/forecasts"
METRICS_DIR = "../outputs/metrics"
FIGURES_DIR = "../outputs/figures"

os.makedirs(FORECAST_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

TARGET = "cnt"
ITEM_ID = "bike_rentals"

FORECAST_HORIZON = 24
VALIDATION_METRIC = "MASE"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load Chronological Data Splits

The train, validation and test sets were created previously during the preprocessing stage. In this notebook, train and validation data will be used for model selection, while the test set will stay untouched until the final evaluation.

In [3]:
train = pd.read_csv(
    f"{DATA_DIR}/train.csv",
    index_col=0,
    parse_dates=True
)

val = pd.read_csv(
    f"{DATA_DIR}/val.csv",
    index_col=0,
    parse_dates=True
)

test = pd.read_csv(
    f"{DATA_DIR}/test.csv",
    index_col=0,
    parse_dates=True
)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

Train shape: (12047, 22)
Validation shape: (3442, 22)
Test shape: (1722, 22)


In [4]:
required_columns = [TARGET]

for split_name, split_df in {
    "train": train,
    "validation": val,
    "test": test
}.items():
    missing_columns = set(required_columns) - set(split_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in {split_name} split: {missing_columns}"
        )

In [5]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(val), len(test)],
    "start": [train.index.min(), val.index.min(), test.index.min()],
    "end": [train.index.max(), val.index.max(), test.index.max()],
    "target_mean": [
        train[TARGET].mean(),
        val[TARGET].mean(),
        test[TARGET].mean()
    ]
})

split_summary

,split,rows,start,end,target_mean
0,train,12047,2011-01-08 07:00:00,2012-05-29 03:00:00,161.749564
1,validation,3442,2012-05-29 04:00:00,2012-10-19 13:00:00,285.973271
2,test,1722,2012-10-19 14:00:00,2012-12-31 23:00:00,203.412892


In [6]:
train.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,...,cnt,lag_1,lag_2,lag_3,lag_24,lag_48,lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168
datetime,,,,,,,,,,,,,,,,,,,,,
2011-01-08 07:00:00,2011-01-08,1,0,1,7,0,6,0,2,0.16,...,9,2.0,5.0,1.0,84.0,36.0,16.0,63.208333,55.844488,56.458333
2011-01-08 08:00:00,2011-01-08,1,0,1,8,0,6,0,3,0.16,...,15,9.0,2.0,5.0,210.0,95.0,40.0,60.083333,56.721989,56.416667
2011-01-08 09:00:00,2011-01-08,1,0,1,9,0,6,0,3,0.16,...,20,15.0,9.0,2.0,134.0,219.0,32.0,51.958333,47.536237,56.267857
2011-01-08 10:00:00,2011-01-08,1,0,1,10,0,6,0,2,0.18,...,61,20.0,15.0,9.0,63.0,122.0,13.0,47.208333,44.585998,56.196429
2011-01-08 11:00:00,2011-01-08,1,0,1,11,0,6,0,2,0.20,...,62,61.0,20.0,15.0,67.0,45.0,1.0,47.125000,44.557059,56.482143
